In [ ]:
import os
import sys
path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(path)
print(path)

from utility import*

import numpy as np
import random
from collections import Counter
import os

In [ ]:

def generate_traffic_skewed(Nrack, Hosts_p_rack, loadfrac0, totaltime, Gbps_rate, Nactive, workload, network, theta, phi, seedValue):
    print(f'Running Python function with arguments: {Nrack}, {Hosts_p_rack}, {loadfrac0}, {totaltime}, {Gbps_rate}, {Nactive}, {workload}, {network}, {seedValue}')

    filename = f'{network}/{workload}_{100 * loadfrac0:.2f}percLoad_{int(totaltime)}sec_{Nrack}N_{Hosts_p_rack}hpr_{Nrack * Hosts_p_rack}hosts_{Gbps_rate}Gbps_{Nactive:.2f}Nactive_{theta:.2f}theta_{phi:.2f}phi_seed={seedValue}.htsim'

    if os.path.exists(filename):
        print(f"File {filename} already exists. Skipping the process.")
        return
    
    np.random.seed(seedValue)
    random.seed(seedValue) 

    H_active = int(np.ceil(Nrack * Hosts_p_rack * Nactive))
    print(f'H_active = {H_active}')

    probabilities, srcdst = get_skewed_probabilities(Nrack, H_active, Hosts_p_rack, theta, phi)
    flowmat1 = get_flow_mat(probabilities, srcdst, workload, Gbps_rate, loadfrac0, H_active, totaltime)

    # Write the flowmat1 to the file in the appropriate format
    write_to_htsim_file(flowmat1, filename)



def get_skewed_probabilities(Nrack, H_active, Hosts_p_rack, theta, phi):

    Ncons = H_active * (H_active - Hosts_p_rack)  # number of possible connections
    srcdst = np.zeros((Ncons, 2), dtype=int)
    
    # Initialize the probability list
    probabilities = []

    num_hot_rack = np.ceil(theta*Nrack)
    theta = num_hot_rack/Nrack

    print(f"num_hot_rack = {num_hot_rack}")
    print(f"new theta = {theta}")
    
    p_hothot = (phi/theta)**2
    p_coldcold = ((1-phi)/(1-theta))**2
    p_hotcold = (phi/theta)*(1-phi)/(1-theta)

    cnt = 0
    print(f"prob = {p_hothot} {p_coldcold} {p_hotcold}")
    for a in range(H_active):  # sources
        for b in range(H_active):  # destinations
            if a // Hosts_p_rack != b // Hosts_p_rack:

                # Store the source-destination pair
                srcdst[cnt] = [a, b]
                if a // Hosts_p_rack < num_hot_rack and b // Hosts_p_rack < num_hot_rack:
                    probabilities.append(p_hothot)
                elif a // Hosts_p_rack >= num_hot_rack and b // Hosts_p_rack >= num_hot_rack:
                    probabilities.append(p_coldcold)
                else:
                    probabilities.append(p_hotcold)
                cnt += 1

    return probabilities, srcdst



In [ ]:
network_ = "clos"

if network_ == "clos":
    Nrack_ = 72
    Hosts_p_rack_ = 9

elif network_ == "opera":
    Nrack_ = 108
    Hosts_p_rack_ = 6

workload_ = "HD"
load_ = 0.05
time_ = 10.001
Gbps_rate_ = 40
Nactive_ = 1
phi_ = 0.80

seeds = [1, 2, 3, 4 ,5]
theta_set = [0.10, 0.20, 0.30, 0.40, 0.50]
for theta_ in theta_set:
    for seedValue_ in seeds:
        generate_traffic_skewed(Nrack_, Hosts_p_rack_, load_, time_, Gbps_rate_, Nactive_, workload_, network_, theta_, phi_, seedValue_)
